In [4]:
from pathlib import Path
from transformers import AutoTokenizer

In [5]:
TOKENIZER_DIR = Path("../tokenizer")
tok = AutoTokenizer.from_pretrained(TOKENIZER_DIR)

print(f"Vocab size : {tok.vocab_size}")
print(f"BOS token  : {tok.bos_token!r} (id={tok.bos_token_id})")
print(f"EOS token  : {tok.eos_token!r} (id={tok.eos_token_id})")
print(f"PAD token  : {tok.pad_token!r} (id={tok.pad_token_id})")

print("\nAll added special tokens:")
for token, idx in sorted(tok.get_added_vocab().items(), key=lambda x: x[1]):
    print(f"  {idx:3d}: {token}")

Vocab size : 50304
BOS token  : '<|bos|>' (id=1)
EOS token  : '<|endoftext|>' (id=0)
PAD token  : '<|pad|>' (id=2)

All added special tokens:
    0: <|endoftext|>
    1: <|bos|>
    2: <|pad|>
    3: <|im_start|>
    4: <|im_end|>
    5: <|tool_call|>
    6: <|/tool_call|>
    7: <|tool_response|>
    8: <|/tool_response|>
    9: <|think|>
   10: <|/think|>


In [6]:
def inspect(label: str, text: str) -> None:
    ids = tok.encode(text, add_special_tokens=False)
    pieces = [tok.decode([i]) for i in ids]
    print(f"=== {label} ({len(ids)} tokens, {len(text)/len(ids):.2f} chars/tok) ===")
    print(" | ".join(repr(p) for p in pieces))
    print("IDs:", ids)
    print()

In [9]:
code_samples = {
    "Python": (
        "@dataclass\n"
        "class RMSNorm(nn.Module):\n"
        "    dim: int\n"
        "    eps: float = 1e-6\n\n"
        "    def forward(self, x: torch.Tensor) -> torch.Tensor:\n"
        "        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)\n"
    ),
    "TypeScript": (
        "export async function fetchUser(id: string): Promise<User | null> {\n"
        "  const res = await fetch(`/api/users/${id}`);\n"
        "  return res.ok ? (await res.json()) : null;\n"
        "}\n"
    ),
    "C++": (
        "#include <vector>\n"
        "int sum(const std::vector<int>& v) {\n"
        "    int acc = 0;\n"
        "    for (int x : v) acc += x;\n"
        "    return acc;\n"
        "}\n"
    ),
    "SQL": (
        "SELECT user_id, COUNT(*) AS total\n"
        "FROM events\n"
        "WHERE created_at >= '2026-01-01'\n"
        "GROUP BY user_id\n"
        "ORDER BY total DESC LIMIT 10;"
    ),
    "Bash": "find src/ -name '*.py' -type f | xargs grep -n 'load_dataset' | head -n 20",
    "JSON": '{"name": "search_docs", "arguments": {"query": "RoPE scaling", "top_k": 5, "exact": false}}',
}

for label, text in code_samples.items():
    inspect(label, text)

=== Python (66 tokens, 3.09 chars/tok) ===
'@' | 'dat' | 'aclass' | '\n' | 'class' | ' R' | 'MS' | 'Norm' | '(nn' | '.Module' | '):\n' | '   ' | ' dim' | ':' | ' int' | '\n' | '   ' | ' eps' | ':' | ' float' | ' =' | ' ' | '1' | 'e' | '-' | '6' | '\n\n' | '   ' | ' def' | ' forward' | '(self' | ',' | ' x' | ':' | ' torch' | '.Tensor' | ')' | ' ->' | ' torch' | '.Tensor' | ':\n' | '       ' | ' return' | ' x' | ' *' | ' torch' | '.rs' | 'qrt' | '(x' | '.pow' | '(' | '2' | ').' | 'mean' | '(-' | '1' | ',' | ' keep' | 'dim' | '=True' | ')' | ' +' | ' self' | '.e' | 'ps' | ')\n'
IDs: [42, 12928, 45057, 209, 1859, 438, 4910, 19539, 41476, 35509, 1052, 310, 6847, 36, 512, 209, 310, 27416, 36, 6213, 399, 231, 27, 79, 23, 32, 490, 310, 760, 4593, 1348, 22, 1703, 36, 8673, 28680, 19, 4669, 8673, 28680, 535, 365, 770, 1703, 606, 8673, 37360, 21723, 4825, 38929, 18, 28, 649, 15589, 10662, 27, 22, 1875, 16462, 4044, 19, 973, 727, 2888, 515, 433]

=== TypeScript (43 tokens, 3.77 chars/tok) ===
'exp

In [10]:
chat_sample = (
    "<|bos|><|im_start|>user\n"
    "What is 17 * 23?<|im_end|>\n"
    "<|im_start|>assistant\n"
    "<|think|>17 * 20 = 340, 17 * 3 = 51, 340 + 51 = 391.<|/think|>\n"
    "<|tool_call|>{\"expr\": \"17 * 23\"}<|/tool_call|>\n"
    "<|tool_response|>391<|/tool_response|>\n"
    "The answer is 391.<|im_end|><|endoftext|>"
)

inspect("Chat + Tool + Think", chat_sample)

=== Chat + Tool + Think (89 tokens, 2.96 chars/tok) ===
'<|bos|>' | '<|im_start|>' | 'user' | '\n' | 'What' | ' is' | ' ' | '1' | '7' | ' *' | ' ' | '2' | '3' | '?' | '<|im_end|>' | '\n' | '<|im_start|>' | 'assistant' | '\n' | '<|think|>' | '1' | '7' | ' *' | ' ' | '2' | '0' | ' =' | ' ' | '3' | '4' | '0' | ',' | ' ' | '1' | '7' | ' *' | ' ' | '3' | ' =' | ' ' | '5' | '1' | ',' | ' ' | '3' | '4' | '0' | ' +' | ' ' | '5' | '1' | ' =' | ' ' | '3' | '9' | '1' | '.' | '<|/think|>' | '\n' | '<|tool_call|>' | '{"' | 'expr' | '":' | ' "' | '1' | '7' | ' *' | ' ' | '2' | '3' | '"}' | '<|/tool_call|>' | '\n' | '<|tool_response|>' | '3' | '9' | '1' | '<|/tool_response|>' | '\n' | 'The' | ' answer' | ' is' | ' ' | '3' | '9' | '1' | '.' | '<|im_end|>' | '<|endoftext|>'
IDs: [1, 3, 2298, 209, 1974, 328, 231, 27, 33, 606, 231, 28, 29, 41, 4, 209, 3, 46217, 209, 9, 27, 33, 606, 231, 28, 26, 399, 231, 29, 30, 26, 22, 231, 27, 33, 606, 231, 29, 399, 231, 31, 27, 22, 231, 29, 30, 26, 973, 231, 31, 27, 3